# Projeto Final de AVD — Parte Específica: The Complete Pokémon Dataset
## Análise Exploratória de Dados (EDA), Agrupamento com K-Means e Avaliação de Métricas

Este notebook compõe a **Parte Específica** do Projeto Final de Análise e Visualização de Dados (AVD — 2026.2).
O dataset utilizado é o **The Complete Pokemon Dataset** (Rounak Banik / Kaggle), contendo informações completas sobre 801 Pokémons das Gerações 1 a 7.

### Objetivos do Estudo:
1. **Exploração e Limpeza de Dados:** Compreender a distribuição dos atributos base de combate (HP, Attack, Defense, Sp. Atk, Sp. Def, Speed e Base Total), tipos e características dos Pokémons.
2. **Engenharia e Pré-processamento:** Tratar valores ausentes e padronizar as variáveis numéricas com `StandardScaler`.
3. **Agrupamento com K-Means:** Descobrir arquétipos naturais de combate (ex: Tanques Defensivos, Atacantes Rápidos, Estágios Iniciais e Lendários/Titãs).
4. **Métricas de Validação:** Avaliar a qualidade dos agrupamentos utilizando Método do Cotovelo (Inércia), Coeficiente de Silhueta e Índice Davies-Bouldin.
5. **Interpretação e Visualização:** Reduzir a dimensionalidade com PCA e traçar o perfil de cada cluster.

In [ ]:
# 1. Importação das bibliotecas essenciais
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Configurações visuais
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 2. Carregamento dos Dados
Carregamos o dataset localizado em `../../data/parte_especifica/pokemon.csv`.

In [ ]:
# 2. Leitura do arquivo
data_path = os.path.join('..', '..', 'data', 'parte_especifica', 'pokemon.csv')
df = pd.read_csv(data_path)
print(f'Dimensões do dataset: {df.shape[0]} linhas e {df.shape[1]} colunas.')
df.head()

## 3. Inspeção e Limpeza de Dados
Verificamos valores ausentes e os tipos das variáveis.

In [ ]:
# 3. Informações gerais e valores ausentes
print(df.info())
null_counts = df.isnull().sum()
print('\nColunas com valores ausentes:')
print(null_counts[null_counts > 0])

In [ ]:
# Tratamento de nulos em altura e peso e padronização de textos
df['height_m'] = pd.to_numeric(df['height_m'], errors='coerce')
df['weight_kg'] = pd.to_numeric(df['weight_kg'], errors='coerce')
df['height_m'] = df['height_m'].fillna(df['height_m'].median())
df['weight_kg'] = df['weight_kg'].fillna(df['weight_kg'].median())
df['type2'] = df['type2'].fillna('None')

# Estatísticas resumo das variáveis de combate
combat_stats = ['hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed', 'base_total']
df[combat_stats].describe().round(2)

## 4. Análise Exploratória de Dados (EDA)
Exploramos distribuições univariadas e bivariadas das estatísticas dos Pokémons.

In [ ]:
# 4.1 Distribuição dos Atributos de Combate (Histogramas)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
stats_ind = ['hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed']

for ax, stat in zip(axes.flatten(), stats_ind):
    sns.histplot(df[stat], kde=True, ax=ax, color='#1d3557', bins=25)
    ax.set_title(f'Distribuição de {stat.upper()}', fontweight='bold')
    ax.set_xlabel('Valor')
    ax.set_ylabel('Frequência')

plt.tight_layout()
plt.show()

In [ ]:
# 4.2 Distribuição de Pokémons por Tipo Primário
plt.figure(figsize=(12, 5))
order_types = df['type1'].value_counts().index
sns.countplot(data=df, x='type1', order=order_types, palette='tab20')
plt.title('Frequência de Pokémons por Tipo Primário (Type 1)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.xlabel('Tipo Primário')
plt.ylabel('Quantidade')
plt.show()

In [ ]:
# 4.3 Matriz de Correlação entre Atributos
plt.figure(figsize=(9, 7))
corr = df[combat_stats + ['height_m', 'weight_kg']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Matriz de Correlação de Pearson entre Atributos', fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# 4.4 Dispersão: Ataque vs Defesa colorido por Status Lendário
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x='attack',
    y='defense',
    hue='is_legendary',
    size='base_total',
    sizes=(20, 200),
    palette={0: '#457b9d', 1: '#e63946'},
    alpha=0.8
)
plt.title('Ataque vs. Defesa (Destaque para Lendários)', fontsize=14, fontweight='bold')
plt.xlabel('Attack')
plt.ylabel('Defense')
plt.legend(title='Lendário', labels=['Comum (0)', 'Lendário (1)'])
plt.show()

## 5. Pré-processamento e Agrupamento com K-Means
Selecionamos as 6 variáveis fundamentais de combate (`hp`, `attack`, `defense`, `sp_attack`, `sp_defense`, `speed`).
Aplicamos o `StandardScaler` para que todas tenham média 0 e variância unitária.

In [ ]:
# 5.1 Seleção de features e Padronização
features = ['hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed']
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Dados padronizados com sucesso. Shape:', X_scaled.shape)

### 5.2 Avaliação do Número Ótimo de Clusters (Elbow, Silhueta e Davies-Bouldin)

In [ ]:
k_values = range(2, 9)
inertias = []
silhouettes = []
davies_bouldins = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
    davies_bouldins.append(davies_bouldin_score(X_scaled, labels))

# Gráficos de Avaliação
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Curva do Cotovelo
ax1.plot(k_values, inertias, 'o-', color='#e63946', linewidth=2)
ax1.set_title('Método do Cotovelo (Elbow Curve)', fontweight='bold')
ax1.set_xlabel('Número de Clusters (k)')
ax1.set_ylabel('Inércia (WCSS)')
ax1.set_xticks(k_values)

# Coeficiente de Silhueta e Davies-Bouldin
ax2.plot(k_values, silhouettes, 's-', color='#1d3557', label='Silhouette (Maior melhor)', linewidth=2)
ax2.set_title('Coeficiente de Silhueta por k', fontweight='bold')
ax2.set_xlabel('Número de Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_xticks(k_values)
ax2.legend()

plt.tight_layout()
plt.show()

for k, s, db in zip(k_values, silhouettes, davies_bouldins):
    print(f'k = {k} | Silhouette Score: {s:.4f} | Davies-Bouldin Index: {db:.4f}')

## 6. Ajuste do Modelo Final (k = 4) e Análise dos Perfis
Com base na análise de cotovelo, silhueta e interpretabilidade de domínio no universo Pokémon, definimos $k = 4$ como um particionamento robusto e bem interpretável.

In [ ]:
# 6.1 Treinamento do K-Means final com k = 4
kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans_final.fit_predict(X_scaled)

# Redução de dimensionalidade com PCA (2 componentes) para visualização
pca = PCA(n_components=2, random_state=42)
pca_res = pca.fit_transform(X_scaled)
df['pca1'] = pca_res[:, 0]
df['pca2'] = pca_res[:, 1]

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=df,
    x='pca1',
    y='pca2',
    hue='cluster',
    palette='tab10',
    alpha=0.85,
    s=60
)
plt.title('Visualização dos Clusters K-Means no Espaço Latente PCA', fontsize=14, fontweight='bold')
plt.xlabel(f'PCA 1 ({pca.explained_variance_ratio_[0]*100:.1f}% da variância)')
plt.ylabel(f'PCA 2 ({pca.explained_variance_ratio_[1]*100:.1f}% da variância)')
plt.legend(title='Cluster')
plt.show()

In [ ]:
# 6.2 Perfil Médio de Atributos por Cluster
cluster_profile = df.groupby('cluster')[features + ['base_total', 'is_legendary']].mean()
cluster_profile['count'] = df.groupby('cluster').size()
cluster_profile['pct_legendary'] = (cluster_profile['is_legendary'] * 100).round(1)
print(cluster_profile.round(1))

# Identificando arquétipos e nomes sugeridos para cada cluster
print('\n--- Interpretação dos Arquétipos dos Clusters ---')
for c in range(4):
    top_pkms = df[df['cluster'] == c]['name'].head(4).tolist()
    b_mean = cluster_profile.loc[c, 'base_total']
    print(f'Cluster {c}: Base Total Média = {b_mean:.1f} | Exemplos: {top_pkms}')

## 7. Conclusões e Integração com o Dashboard

1. **Segmentação Natural:** Os 4 clusters representam com clareza:
   - **Cluster de Estágios Iniciais / Baixo Poder:** Pokémons básicos com atributos reduzidos em todas as áreas.
   - **Cluster de Tanques / Defensores:** Elevado HP e Defense, velocidade reduzida.
   - **Cluster de Atacantes Ágeis (Sweepers):** Alto Attack/Sp. Atk e Speed.
   - **Cluster de Titãs / Lendários:** Pokémons com estatísticas superiores e balanceadas, concentrando a grande maioria dos lendários.
2. **Dashboard Interativo:** Todos esses agrupamentos, gráficos e explorações foram integrados no aplicativo interativo construído em Streamlit (`src/parte_especifica/dashboard.py`), permitindo controle dinâmico de hiperparâmetros, filtros de tipos e comparador visual.